# 🌾 RICE VISION AI PIPELINE V2: DỰ ĐOÁN HỒI QUY THUẦN (YOLOv8m-SEG SOTA)
### 🌟 Các cải tiến chính so với Pipeline V1:
1. **Tích hợp mô hình YOLOv8m-seg (SOTA)**: Đạt Mask mAP cao nhất (85.85%) và Box mAP cao nhất (91.50%), bóc tách hạt lúa chính xác nhất.
2. **📸 Hệ thống lưu ảnh hạt đa tầng & So sánh trực quan tại `/content/grain_crops/`**:
   - `0_raw_before_clean/`: Toàn bộ hạt cắt thô từ YOLO (TRƯỚC KHI CLEAN).
   - `1_after_clean/`: Toàn bộ hạt sau khi qua 2 lần clean chuyên sâu (SAU KHI CLEAN).
   - `2_after_cnn_filter/`: Tách riêng `hat_nguyen` (đạt chuẩn) và `hat_khuyet_tat` (bị loại).
   - `3_after_iqr_filter/`: Tách riêng `kept_grains` (đạt chuẩn cuối cùng) và `rejected_grains` (outlier nhỏ).
   - 🔍 **Side-by-Side Gallery**: Xem trực quan cạnh nhau ảnh hạt TRƯỚC vs SAU clean ngay trong notebook.
3. **Bỏ hoàn toàn dự đoán vật lý (Packing Fraction)**: Không dùng hệ số chèn lấp; tính **Tỷ số thể tích thuần** $V_{bulk} / \bar{V}_{grain}$ làm đặc trưng để dự đoán trực tiếp bằng Hồi quy Machine Learning.
4. **Tích hợp tham chiếu Cân nặng (Weight)**: Hỗ trợ kiểm chứng số hạt từ cảm biến cân.

In [ ]:
# Bước 1: Kết nối Google Drive & Cài đặt thư viện
import os
import sys

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
    print('✅ Đã kết nối Google Drive thành công!')
    !pip install -q ultralytics sahi tensorflow matplotlib seaborn opencv-python numpy pandas joblib scikit-learn==1.6.1
except ImportError:
    print('💻 Đang chạy trên môi trường cục bộ (Local)')


In [ ]:
# Bước 2: Import Modules, Kiểm Tra GPU & Nạp Trọng Số Model (YOLOv8m & CNN)
import os
import sys
import math
import shutil
import importlib
import torch
import tensorflow as tf
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D

# CẤU HÌNH ĐƯỜNG DẪN DỰ ÁN TRÊN GOOGLE DRIVE / LOCAL
BASE_PATH = "/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES"
if not os.path.exists(BASE_PATH):
    alt_path = "/content/drive/MyDrive/NGHIEN CUU KHOA HOC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES"
    if os.path.exists(alt_path):
        BASE_PATH = alt_path
    elif not os.path.exists("/content"):
        BASE_PATH = os.path.abspath(".") if os.path.exists("CODE") else os.path.abspath(os.path.join(os.getcwd(), ".."))

code_dir = os.path.join(BASE_PATH, "AI_SERVICES/src")
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)
if BASE_PATH not in sys.path:
    sys.path.insert(0, BASE_PATH)

# Import các module nghiệp vụ cốt lõi
from rice_ai.vision.container_detector import detect_container_and_scale, draw_container_overlay
from rice_ai.vision.grain_segmenter import segment_grains_sahi
from rice_ai.vision.grain_crop_cleaner import clean_single_grain_crop
from rice_ai.vision.grain_classifier import GrainClassifier
from rice_ai.vision.ellipsoid_geometry import compute_single_grain_metrics, draw_grain_ellipse_overlay
from rice_ai.vision.grain_size_filter import filter_grains_by_size
from rice_ai.vision.uniformity_evaluator import evaluate_batch_uniformity

# Kiểm tra GPU
gpu_available = torch.cuda.is_available()
device_str = "cuda:0" if gpu_available else "cpu"
gpu_name = torch.cuda.get_device_name(0) if gpu_available else "CPU (Không có GPU)"
print("=" * 75)
print(f"⚡ Trạng thái GPU (PyTorch)   : {'✅' if gpu_available else '⚠️'} {gpu_name} ({device_str})")
print("=" * 75)

# 1. ĐƯỜNG DẪN MÔ HÌNH YOLOV8M-SEG (SOTA MASK MAP = 85.85%, BOX MAP = 91.50%)
YOLO_MODEL_PATH = os.path.join(BASE_PATH, "RESULTS/Segmentation Model Results/yolov8m/weights/best.pt")
if not os.path.exists(YOLO_MODEL_PATH):
    candidates = [
        os.path.join(BASE_PATH, "RESULTS/Segmentation Model Results/yolov26s/weights/best.pt"),
        os.path.join(BASE_PATH, "RESULTS/Segmentation Model Results/yolov8s/weights/best.pt"),
    ]
    for c in candidates:
        if os.path.exists(c):
            YOLO_MODEL_PATH = c
            break

# 2. ĐƯỜNG DẪN MÔ HÌNH CNN DENSENET121
CNN_MODEL_PATH = os.path.join(BASE_PATH, "RESULTS/CNN_DenseNet121_Trained/best_v3_step2.keras")

print(f"🔹 YOLO Model : {YOLO_MODEL_PATH}")
print(f"🔹 CNN Model  : {CNN_MODEL_PATH}")

# Nạp SAHI YOLO model
from sahi import AutoDetectionModel
print(f"⏳ Đang nạp mô hình SAHI YOLO-seg trên [{device_str}]...")
sahi_yolo_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=YOLO_MODEL_PATH,
    confidence_threshold=0.50,
    device=device_str
)

# Nạp CNN model
print(f"⏳ Đang nạp mô hình DenseNet121...")
cnn_classifier = GrainClassifier(
    model_path=CNN_MODEL_PATH,
    class_names=["hat_khuyet_tat", "hat_nguyen"],
    target_size=(224, 224)
)
print("✅ Nạp toàn bộ mô hình AI thành công!")


In [ ]:
# Bước 3: THIẾT LẬP ĐƯỜNG DẪN ẢNH, THÔNG SỐ VẬT CHỨA VÀ THƯ MỤC LƯU ẢNH HẠT
# ─────────────────────────────────────────────────────────────────────────────

# 1. ĐƯỜNG DẪN ẢNH ĐẦU VÀO CẦN DỰ ĐOÁN
IMAGE_PATH = "/content/M0301.jpg"

if not os.path.exists(IMAGE_PATH):
    fallback_path = os.path.join(BASE_PATH, "DATASET_BUILDER/1_Raw_Images/M030/M0301.jpg")
    if not os.path.exists(fallback_path):
        fallback_path = os.path.join(BASE_PATH, "DATASET_BUILDER/1_Raw_Images/M0001.jpg")
    if os.path.exists(fallback_path):
        IMAGE_PATH = fallback_path
        print(f"⚠️ Dùng ảnh mẫu có sẵn: {IMAGE_PATH}")
    else:
        print(f"⚠️ Vui lòng upload ảnh cần dự đoán lên: {IMAGE_PATH}")

# 2. CHẾ ĐỘ QUÉT SAHI
USE_CROPPED_CONTAINER = False   # False = Mặt lúa trong khung toàn ảnh | True = Ảnh Crop Miệng Ly

# 3. THÔNG SỐ VẬT CHỨA (ĐO THỰC TẾ)
INNER_DIAM_MM         = 41.5    # Đường kính MÉP TRONG miệng ly (mm)
TRUE_RICE_DIAM_MM     = INNER_DIAM_MM    # Đường kính thực tế lõi phễu chứa lúa (mm)
CONTAINER_HEIGHT_MM   = 60.0    # Chiều cao toàn bộ thân ly (mm)
EMPTY_HEIGHT_MM       = 52.8    # Khoảng trống từ miệng ly đến mặt lúa (mm)
INPUT_WEIGHT_G        = 4.23    # Cân nặng mẫu (g)
CONF_THRESHOLD_NGUYEN = 0.90    # Ngưỡng xác suất công nhận là Hạt Nguyên (>= 0.90)

# 4. THAM SỐ 2 LẦN CLEAN
CLEAN_PARAMS_STEP1 = {"open_ksize": 5, "min_neck_ratio": 0.15, "min_area": 35, "centrality_weight": 2.5, "fill_holes": False, "sever_bridges": False}
CLEAN_PARAMS_STEP2 = {"open_ksize": 3, "min_neck_ratio": 0.15, "min_area": 25, "centrality_weight": 2.2, "fill_holes": True, "sever_bridges": False}

# 5. CẤU HÌNH BỘ LỌC KÍCH THƯỚC IQR
ENABLE_SIZE_FILTER = True
SIZE_FILTER_K = 0.10
SIZE_FILTER_MIN_SAMPLES = 8

# 6. CẤU HÌNH THƯ MỤC XUẤT KẾT QUẢ VÀ LƯU ẢNH TỪNG HẠT (THEO YÊU CẦU)
OUTPUT_DIR = "/content/pipeline_inference_results" if os.path.exists("/content") else os.path.join(BASE_PATH, "OUTPUT_RESULTS")
OUTPUT_REPORT_DIR = os.path.join(OUTPUT_DIR, "reports")

# CẤU TRÚC THƯ MỤC LƯU ẢNH HẠT ĐẦY ĐỦ CÁC GIAI ĐOẠN:
OUTPUT_GRAIN_CROPS_ROOT     = "/content/grain_crops" if os.path.exists("/content") else os.path.join(BASE_PATH, "OUTPUT_GRAIN_CROPS")
DIR_STAGE0_RAW_BEFORE_CLEAN = os.path.join(OUTPUT_GRAIN_CROPS_ROOT, "0_raw_before_clean")        # Hạt thô từ YOLO (TRƯỚC CLEAN)
DIR_STAGE1_AFTER_CLEAN      = os.path.join(OUTPUT_GRAIN_CROPS_ROOT, "1_after_clean")            # Hạt sau Clean 2 bước (SAU CLEAN)
DIR_STAGE2_CNN_WHOLE        = os.path.join(OUTPUT_GRAIN_CROPS_ROOT, "2_after_cnn_filter/hat_nguyen")
DIR_STAGE2_CNN_DEFECT       = os.path.join(OUTPUT_GRAIN_CROPS_ROOT, "2_after_cnn_filter/hat_khuyet_tat")
DIR_STAGE3_IQR_KEPT         = os.path.join(OUTPUT_GRAIN_CROPS_ROOT, "3_after_iqr_filter/kept_grains")
DIR_STAGE3_IQR_REJECTED     = os.path.join(OUTPUT_GRAIN_CROPS_ROOT, "3_after_iqr_filter/rejected_grains")

ALL_GRAIN_DIRS = [
    OUTPUT_DIR, OUTPUT_REPORT_DIR,
    DIR_STAGE0_RAW_BEFORE_CLEAN, DIR_STAGE1_AFTER_CLEAN,
    DIR_STAGE2_CNN_WHOLE, DIR_STAGE2_CNN_DEFECT,
    DIR_STAGE3_IQR_KEPT, DIR_STAGE3_IQR_REJECTED
]
for d in ALL_GRAIN_DIRS:
    os.makedirs(d, exist_ok=True)

print("=" * 70)
print(f"🖼️  Ảnh phân tích          : {IMAGE_PATH}")
print(f"📏 Kích thước ly          : Đường kính={INNER_DIAM_MM}mm | Cao={CONTAINER_HEIGHT_MM}mm | Hụt={EMPTY_HEIGHT_MM}mm")
print(f"⚖️  Khối lượng mẫu         : {INPUT_WEIGHT_G} g")
print(f"📂 Thư mục lưu ảnh hạt    : {OUTPUT_GRAIN_CROPS_ROOT}")
print(f"   ├── 0_raw_before_clean : Ảnh cắt thô từ YOLO (TRƯỚC KHI CLEAN)")
print(f"   ├── 1_after_clean      : Ảnh sau Clean 2 bước (SAU KHI CLEAN)")
print(f"   ├── 2_after_cnn_filter : Tách riêng hat_nguyen / hat_khuyet_tat")
print(f"   └── 3_after_iqr_filter : Tách riêng kept_grains / rejected_grains")
print("=" * 70)


In [ ]:
# Bước 4: TẦNG 1 — Nhận diện Miệng Ly, Tính Thể tích Khối Lúa & Tạo Ảnh Crop
# Xóa trạng thái từ lần chạy trước để detector lỗi không làm downstream dùng dữ liệu cũ.
container_info = None
pixels_per_mm = None
bulk_volume_mm3 = None
cropped_container_bgr = None
SURFACE_FULL_PATH = None

container_info = detect_container_and_scale(
    image_input=IMAGE_PATH,
    inner_diam_mm=INNER_DIAM_MM,
    container_height_mm=CONTAINER_HEIGHT_MM,
    empty_height_mm=EMPTY_HEIGHT_MM,
    detect_mode="inner",
    surface_method="adaptive",
    min_outer_diam_ratio=0.05,
    min_inner_diam_ratio=0.03
)

pixels_per_mm = container_info["pixels_per_mm"]
import math
bulk_volume_mm3 = math.pi * ((TRUE_RICE_DIAM_MM / 2.0)**2) * container_info["rice_height_mm"]
crop_offset_x, crop_offset_y = container_info["crop_offset"]
cropped_container_bgr = container_info["cropped_bgr"]

# Full-frame mask preserves SAHI coordinates while excluding reflected grains.
SURFACE_FULL_PATH = os.path.join(OUTPUT_DIR, "1_surface_full.png")
if not cv2.imwrite(SURFACE_FULL_PATH, container_info["surface_isolated_bgr"]):
    raise RuntimeError("Cannot save the isolated rice surface")
print("Surface diagnostics:", container_info.get("surface_diagnostics"))
for warning in container_info.get("warnings", []):
    print("⚠️", warning)

# Lưu ảnh đã crop cô lập miệng ly ra đĩa
CROPPED_CONTAINER_PATH = os.path.join(OUTPUT_DIR, "1_cropped_container.jpg")
cv2.imwrite(CROPPED_CONTAINER_PATH, cropped_container_bgr)

# Lưu ảnh overlay vẽ khoanh mép trong / mép ngoài
vis_container = container_info["overlay_bgr"]
out_container_img = os.path.join(OUTPUT_DIR, "1_container_detection.jpg")
cv2.imwrite(out_container_img, vis_container)

print("=" * 70)
print("  TẦNG 1: QUY ĐỔI KÍCH THƯỚC VẬT CHỨA (ƯỚC LƯỢNG TỪ ĐƯỜNG BAO MẶT LÚA)")
print("=" * 70)
print(f"  • Loại nhận diện                 : {container_info['detect_type']}")
print(f"  • Đường kính MÉP TRONG phát hiện : {container_info['inner_w_px']} px (Mép ngoài: {container_info['outer_w_px']} px)")
print(f"  • Tỷ lệ quy đổi pixel           : {pixels_per_mm:.2f} px/mm")
print(f"  • Chiều cao khối lúa thực tế    : {container_info['rice_height_mm']:.1f} mm")
print(f"  • Thể tích khối lúa (V_bulk)    : {bulk_volume_mm3:,.2f} mm³")
print(f"  • Tọa độ Crop Bounding Box       : Offset=({crop_offset_x}, {crop_offset_y}) | Size={cropped_container_bgr.shape[1]}x{cropped_container_bgr.shape[0]} px")
print("=" * 70)

# Hiển thị ảnh khoanh viền và ảnh đã cắt gọn miệng ly
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(vis_container, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Ảnh Gốc (Khoanh Mép Trong {pixels_per_mm:.2f} px/mm)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(cropped_container_bgr, cv2.COLOR_BGR2RGB))
axes[1].set_title("Mặt lúa đã tách khỏi vùng phản chiếu", fontweight="bold")
axes[1].axis("off")
plt.tight_layout()
plt.show()
print(f"💾 Đã lưu ảnh cô lập miệng ly vào: {CROPPED_CONTAINER_PATH}")


In [ ]:
if container_info is None or SURFACE_FULL_PATH is None:
    raise RuntimeError("Run container detection successfully before SAHI")

# Bước 5: TẦNG 2 — BÓC TÁCH HẠT BẰNG SAHI YOLO-SEG & LƯU ẢNH THÔ TRƯỚC KHI CLEAN
if USE_CROPPED_CONTAINER:
    sahi_target_image = CROPPED_CONTAINER_PATH
    offset_x, offset_y = crop_offset_x, crop_offset_y
    print(f"⏳ Đang chạy SAHI YOLO-seg trên [ẢNH CROP MIỆNG LY]: {sahi_target_image}")
else:
    sahi_target_image = SURFACE_FULL_PATH
    offset_x, offset_y = 0, 0
    print(f"⏳ Đang chạy SAHI YOLO-seg trên [MẶT LÚA TRONG KHUNG TOÀN ẢNH]: {sahi_target_image}")

raw_grains = segment_grains_sahi(
    detection_model=sahi_yolo_model,
    image_path=sahi_target_image,
    output_crop_dir=None,
    conf_threshold=0.50,
    slice_size=640,
    overlap_ratio=0.25
)

# Dọn dẹp thư mục raw cũ
for f in os.listdir(DIR_STAGE0_RAW_BEFORE_CLEAN):
    fp = os.path.join(DIR_STAGE0_RAW_BEFORE_CLEAN, f)
    if os.path.isfile(fp): os.remove(fp)

# Cập nhật Bounding Box và 📸 LƯU ẢNH HẠT THÔ TRƯỚC KHI CLEAN (TỪ YOLO CẮT RA)
for idx, g in enumerate(raw_grains):
    lx1, ly1, lx2, ly2 = g['bbox']
    g['local_bbox']  = [lx1, ly1, lx2, ly2]
    g['global_bbox'] = [lx1 + offset_x, ly1 + offset_y, lx2 + offset_x, ly2 + offset_y]
    grain_idx = g.get("grain_id", g.get("index", idx))
    
    # Lưu ảnh thô nguyên bản từ YOLO
    raw_save_path = os.path.join(DIR_STAGE0_RAW_BEFORE_CLEAN, f"grain_{grain_idx:04d}_raw_before_clean.png")
    cv2.imwrite(raw_save_path, g["crop_rgba"])
    g["saved_raw_path"] = raw_save_path

print(f"✅ Bóc tách hoàn tất: Phát hiện {len(raw_grains)} hạt lúa thô.")
print(f"📁 📸 ĐÃ LƯU {len(raw_grains)} ẢNH HẠT THÔ (TRƯỚC CLEAN) VÀO: {DIR_STAGE0_RAW_BEFORE_CLEAN}")


In [ ]:
# Bước 6: TẦNG 3 — ÁP DỤNG 2 LẦN CLEAN, LƯU ẢNH SAU CLEAN & XEM SO SÁNH TRỰC QUAN
print(f"🧹 Đang thực hiện 2 lần Clean chuyên sâu cho {len(raw_grains)} hạt lúa...")

# Xóa file cũ trong thư mục sau clean
for f in os.listdir(DIR_STAGE1_AFTER_CLEAN):
    fp = os.path.join(DIR_STAGE1_AFTER_CLEAN, f)
    if os.path.isfile(fp): os.remove(fp)

cleaned_grains = []
discarded_count = 0

for idx, g in enumerate(raw_grains):
    raw_crop_rgba = g["crop_rgba"]
    grain_idx = g.get("grain_id", g.get("index", idx))

    # LẦN 1: Clean Step 1 (Open KSize=5, Min Neck=0.15, Min Area=35)
    crop_step1 = clean_single_grain_crop(raw_crop_rgba, **CLEAN_PARAMS_STEP1)

    # LẦN 2: Clean Step 2 (Open KSize=3, Min Neck=0.15, Min Area=25, Fill=True)
    crop_step2 = clean_single_grain_crop(crop_step1, **CLEAN_PARAMS_STEP2)

    # Kiểm tra mask sau 2 lần clean
    alpha = crop_step2[:, :, 3] if len(crop_step2.shape) == 3 and crop_step2.shape[2] == 4 else None
    if alpha is not None and cv2.countNonZero(alpha) >= 20:
        g_clean = dict(g)
        g_clean["crop_rgba_raw"] = raw_crop_rgba  # Giữ lại ảnh thô để so sánh
        g_clean["crop_rgba"] = crop_step2
        g_clean["grain_id"] = grain_idx
        clean_bgr = crop_step2[:, :, :3].copy()
        clean_bgr[crop_step2[:, :, 3] == 0] = [0, 0, 0]
        g_clean["crop_bgr"] = clean_bgr

        # 📸 LƯU ẢNH SAU KHI CLEAN 2 BƯỚC
        clean_save_path = os.path.join(DIR_STAGE1_AFTER_CLEAN, f"grain_{grain_idx:04d}_after_clean.png")
        cv2.imwrite(clean_save_path, crop_step2)
        g_clean["saved_clean_path"] = clean_save_path
        cleaned_grains.append(g_clean)
    else:
        discarded_count += 1

print("=" * 70)
print("✨ KẾT QUẢ KHỬ NHIỄU 2 LẦN (STEP 1 + STEP 2)")
print("=" * 70)
print(f"• Tổng hạt thô ban đầu (Trước Clean) : {len(raw_grains)} hạt")
print(f"• Hạt lúa sạch giữ lại (Sau Clean)     : {len(cleaned_grains)} hạt")
print(f"• Nhiễu vụn đã loại bỏ (🗑️)             : {discarded_count} mảnh")
print(f"📁 Đã lưu ảnh SAU CLEAN vào           : {DIR_STAGE1_AFTER_CLEAN}")
print("=" * 70)

# ==============================================================================
# 🔍 XEM TRỰC QUAN: SO SÁNH TRƯỚC CLEAN (TRÁI) vs SAU CLEAN (PHẢI)
# ==============================================================================
num_compare = min(8, len(cleaned_grains))
if num_compare > 0:
    fig, axes = plt.subplots(num_compare, 2, figsize=(7, num_compare * 2.6))
    if num_compare == 1:
        axes = np.array([axes])
    for i in range(num_compare):
        g = cleaned_grains[i]
        
        # 1. Ảnh TRƯỚC CLEAN (Raw từ YOLO)
        raw_rgba = g["crop_rgba_raw"]
        raw_alpha = (raw_rgba[:, :, 3] / 255.0) if raw_rgba.shape[2] == 4 else np.ones(raw_rgba.shape[:2])
        raw_rgb = cv2.cvtColor(raw_rgba[:, :, :3], cv2.COLOR_BGR2RGB)
        raw_white = np.ones_like(raw_rgb) * 255
        raw_show = (raw_rgb * raw_alpha[:, :, None] + raw_white * (1 - raw_alpha[:, :, None])).astype(np.uint8)
        
        # 2. Ảnh SAU CLEAN 2 BƯỚC
        clean_rgba = g["crop_rgba"]
        clean_alpha = (clean_rgba[:, :, 3] / 255.0) if clean_rgba.shape[2] == 4 else np.ones(clean_rgba.shape[:2])
        clean_rgb = cv2.cvtColor(clean_rgba[:, :, :3], cv2.COLOR_BGR2RGB)
        clean_white = np.ones_like(clean_rgb) * 255
        clean_show = (clean_rgb * clean_alpha[:, :, None] + clean_white * (1 - clean_alpha[:, :, None])).astype(np.uint8)
        
        axes[i, 0].imshow(raw_show)
        axes[i, 0].set_title(f"Hạt #{g['grain_id']}: TRƯỚC CLEAN (Raw)", fontsize=8, fontweight='bold', color='#D32F2F')
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(clean_show)
        axes[i, 1].set_title(f"Hạt #{g['grain_id']}: SAU CLEAN 2 BƯỚC", fontsize=8, fontweight='bold', color='#388E3C')
        axes[i, 1].axis("off")
        
    plt.suptitle("🔍 SO SÁNH TRỰC QUAN: TRƯỚC CLEAN vs SAU CLEAN 2 BƯỚC", fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
# Bước 7: TẦNG 4 — BƯỚC LỌC 1 (DENSENET121 CNN) & LƯU ẢNH SAU LỌC PHẨM CẤP
print(f"⏳ Đang phân loại {len(cleaned_grains)} hạt bằng DenseNet121 CNN (Batch GPU)...")

# Xóa sạch file cũ trong thư mục phân loại
for folder in [DIR_STAGE2_CNN_WHOLE, DIR_STAGE2_CNN_DEFECT]:
    for f in os.listdir(folder):
        fp = os.path.join(folder, f)
        if os.path.isfile(fp): os.remove(fp)

whole_grains, defective_grains = cnn_classifier.filter_grains(
    cleaned_grains, target_label="hat_nguyen", min_conf=CONF_THRESHOLD_NGUYEN
)

# 📸 LƯU ẢNH GIAI ĐOẠN 2: SAU BƯỚC LỌC 1 (CNN PHẨM CẤP)
for idx, g in enumerate(whole_grains):
    if "crop_rgba" in g and g["crop_rgba"] is not None:
        grain_idx = g.get("grain_id", g.get("index", idx))
        conf = g.get("confidence", 1.0)
        fname = f"grain_{grain_idx:04d}_whole_conf{conf:.2f}.png"
        cv2.imwrite(os.path.join(DIR_STAGE2_CNN_WHOLE, fname), g["crop_rgba"])

for idx, g in enumerate(defective_grains):
    if "crop_rgba" in g and g["crop_rgba"] is not None:
        grain_idx = g.get("grain_id", g.get("index", idx))
        conf = g.get("confidence", 1.0)
        fname = f"grain_{grain_idx:04d}_defect_conf{conf:.2f}.png"
        cv2.imwrite(os.path.join(DIR_STAGE2_CNN_DEFECT, fname), g["crop_rgba"])

all_classified_grains = whole_grains + defective_grains
whole_ratio = (len(whole_grains) / len(cleaned_grains) * 100) if cleaned_grains else 0.0

print("=" * 70)
print("📊 TẦNG 4: KẾT QUẢ SÀNG LỌC HẠT NGUYÊN (BƯỚC LỌC 1: CNN)")
print("=" * 70)
print(f"• Tổng hạt lúa sạch trên mặt ly          : {len(cleaned_grains)} hạt")
print(f"• Hạt NGUYÊN đạt chuẩn (ĐƯỢC CHỌN ĐO 3D) : {len(whole_grains)} hạt ({whole_ratio:.1f}%)")
print(f"• Hạt khuyết tật / lép (Loại khỏi đo 3D)  : {len(defective_grains)} hạt ({100 - whole_ratio:.1f}%)")
print(f"📁 Đã lưu {len(whole_grains)} hạt nguyên vào      : {DIR_STAGE2_CNN_WHOLE}")
print(f"📁 Đã lưu {len(defective_grains)} hạt khuyết tật vào  : {DIR_STAGE2_CNN_DEFECT}")
print("=" * 70)

# Vẽ và lưu biểu đồ tròn phẩm cấp
if cleaned_grains:
    plt.figure(figsize=(5, 5))
    plt.pie(
        [max(1, len(whole_grains)), max(1, len(defective_grains))],
        labels=['Hat nguyen (Do 3D)', 'Hat khuyet tat (Loai)'],
        autopct='%.1f%%',
        colors=['#4CAF50', '#F44336'],
        startangle=140,
        explode=(0.05, 0)
    )
    plt.title(f"Ty Le Sang Loc Hat Nguyen (Tong: {len(cleaned_grains)} hat)", fontweight='bold')
    pie_chart_path = os.path.join(OUTPUT_REPORT_DIR, "2_classification_pie_chart.png")
    plt.savefig(pie_chart_path, dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
# ==============================================================================
# BƯỚC 8: TẦNG 5 — GRAIN_MEASUREMENTS (ĐO ĐẠC HÌNH HỌC 3D ELLIPSOID)
# ==============================================================================
whole_grain_metrics_list = []
whole_records = []
whole_volumes_list = []
raw_valid_pairs = []
measurement_errors = []

if not np.isfinite(pixels_per_mm) or pixels_per_mm <= 0:
    raise ValueError("pixels_per_mm phải hữu hạn và > 0; chạy lại bước nhận diện miệng ly.")

grains_to_measure_3d = whole_grains if len(whole_grains) > 0 else []

for idx, g in enumerate(grains_to_measure_3d):
    try:
        m = compute_single_grain_metrics(
            g["crop_rgba"], pixels_per_mm=pixels_per_mm, label="hat_nguyen"
        )
        for key in ("length_mm", "width_mm", "thickness_mm", "area_mm2", "volume_mm3"):
            if not np.isfinite(m[key]) or m[key] <= 0:
                raise ValueError(f"Số đo {key} không hợp lệ: {m[key]}")
        grain_idx = g.get("grain_id", g.get("index", idx))
        rec = {
            "candidate_index": idx,
            "grain_id": grain_idx,
            "label": "hat_nguyen",
            "confidence": float(g.get("confidence", 1.0)),
            "length_mm_2a": round(float(m["length_mm"]), 3),
            "width_mm_2b": round(float(m["width_mm"]), 3),
            "thickness_mm_2c": round(float(m["thickness_mm"]), 3),
            "area_mm2": round(float(m["area_mm2"]), 3),
            "volume_3d_mm3": round(float(m["volume_mm3"]), 3)
        }
        whole_grain_metrics_list.append(m)
        whole_volumes_list.append(m["volume_mm3"])
        whole_records.append(rec)
        raw_valid_pairs.append({"grain": g, "record": rec, "metrics": m})
    except Exception as e:
        measurement_errors.append({
            "candidate_index": idx, "grain_id": g.get("grain_id", g.get("index", idx)),
            "error": f"{type(e).__name__}: {e}",
        })

print(f"✅ Đo hình học: {len(whole_grains)} hạt CNN chọn nguyên → {len(whole_records)} hợp lệ, {len(measurement_errors)} lỗi.")


In [ ]:
# ==============================================================================
# BƯỚC 9: TẦNG 5.1 — BƯỚC LỌC 2 (BỘ LỌC KÍCH THƯỚC IQR) & LƯU ẢNH GIAI ĐOẠN 3
# ==============================================================================
print(f"🔄 Đang lọc hạt outlier diện tích nhỏ... (Enabled={ENABLE_SIZE_FILTER})")

# Dọn dẹp thư mục sau lọc 2
for folder in [DIR_STAGE3_IQR_KEPT, DIR_STAGE3_IQR_REJECTED]:
    for f in os.listdir(folder):
        fp = os.path.join(folder, f)
        if os.path.isfile(fp): os.remove(fp)

filter_result = filter_grains_by_size(
    measurements=whole_records,
    k=SIZE_FILTER_K,
    min_samples=SIZE_FILTER_MIN_SAMPLES,
    enabled=ENABLE_SIZE_FILTER
)

filter_stats = filter_result["stats"]
kept_records = filter_result["kept"]
rejected_records = filter_result["rejected"]

filtered_whole_records = []
filtered_whole_grains = []
filtered_whole_grain_metrics_list = []
filtered_whole_volumes_list = []
filtered_pairs = []

kept_indices = {rec["candidate_index"] for rec in kept_records}
for pair in raw_valid_pairs:
    c_idx = pair["record"]["candidate_index"]
    g_id = pair["record"]["grain_id"]
    crop_rgba = pair["grain"]["crop_rgba"]
    vol = pair["record"]["volume_3d_mm3"]
    area = pair["record"]["area_mm2"]

    if c_idx in kept_indices:
        filtered_whole_records.append(pair["record"])
        filtered_whole_grains.append(pair["grain"])
        filtered_whole_grain_metrics_list.append(pair["metrics"])
        filtered_whole_volumes_list.append(vol)
        filtered_pairs.append(pair)

        # 📸 LƯU HẠT ĐẠT CHUẨN CUỐI CÙNG
        cv2.imwrite(os.path.join(DIR_STAGE3_IQR_KEPT, f"grain_{g_id:04d}_kept_area{area:.1f}.png"), crop_rgba)
    else:
        # 📸 LƯU HẠT BỊ LOẠI BỞI IQR
        cv2.imwrite(os.path.join(DIR_STAGE3_IQR_REJECTED, f"grain_{g_id:04d}_rejected_area{area:.1f}.png"), crop_rgba)

print(f"📊 Trạng thái lọc: {filter_stats['status']}")
print(f"📊 Kết quả Lọc  : Giữ lại {filter_stats['count_after']}/{filter_stats['count_before']} hạt.")
if filter_stats.get('lower_bound') is not None:
    print(f"   (Area Lower Bound: {filter_stats['lower_bound']:.2f} mm2)")

print(f"📁 Đã lưu {len(filtered_whole_records)} hạt đạt chuẩn vào : {DIR_STAGE3_IQR_KEPT}")
print(f"📁 Đã lưu {len(rejected_records)} hạt bị loại vào       : {DIR_STAGE3_IQR_REJECTED}")

# Xuất CSV quyết định lọc
df_decisions = pd.DataFrame(kept_records + rejected_records)
if not df_decisions.empty:
    csv_decision_path = os.path.join(OUTPUT_REPORT_DIR, "grain_size_filter_decisions.csv")
    df_decisions.to_csv(csv_decision_path, index=False, encoding="utf-8-sig")


In [ ]:
# ==============================================================================
# BƯỚC 10: TẦNG 5.2 — TỶ SỐ THỂ TÍCH THUẦN (V_bulk / V_grain) — KHÔNG DÙNG PACKING FRACTION
# ==============================================================================
if not filtered_whole_volumes_list:
    print("⚠️ Không đủ dữ liệu hợp lệ sau lọc.")
    mean_whole_grain_vol = None
    pure_volume_ratio = None
else:
    uniformity_res_raw = evaluate_batch_uniformity(whole_volumes_list, threshold_rate=0.80)
    uniformity_res = evaluate_batch_uniformity(filtered_whole_volumes_list, threshold_rate=0.80)
    mean_whole_grain_vol = uniformity_res["mean_clean"]

    # 🔥 TỶ SỐ THỂ TÍCH THUẦN: V_bulk / V_grain (Mặc kệ hệ số lắp đầy)
    pure_volume_ratio = (bulk_volume_mm3 / mean_whole_grain_vol) if (mean_whole_grain_vol and mean_whole_grain_vol > 0) else 0.0

    print("=" * 75)
    print("  TẦNG 5.2: KẾT QUẢ ĐO KÍCH THƯỚC 3D & TỶ SỐ THỂ TÍCH THUẦN (KHÔNG DÙNG PACKING)")
    print("=" * 75)
    lengths = [r["length_mm_2a"] for r in filtered_whole_records]
    widths  = [r["width_mm_2b"] for r in filtered_whole_records]
    thick   = [r["thickness_mm_2c"] for r in filtered_whole_records]
    areas   = [r["area_mm2"] for r in filtered_whole_records]
    vols    = [r["volume_3d_mm3"] for r in filtered_whole_records]

    print(f"  • Số hạt sau bộ lọc IQR           : {len(filtered_whole_records)} hạt (từ {len(whole_records)} hạt đo đạc ban đầu)")
    print(f"  • Thể tích hạt trung bình raw     : {uniformity_res_raw['mean_clean']:.2f} mm³")
    print(f"  • Thể tích hạt trung bình sạch    : {mean_whole_grain_vol:.2f} mm³ (sau lọc IQR)")
    print(f"  • Độ đồng đều lô hạt             : {uniformity_res['uniformity_rate_pct']}% -> {'✅ ĐẠT' if uniformity_res['is_uniform'] else '❌ KHÔNG ĐẠT'}")
    print("-" * 75)
    print(f"  🔥 Thể tích khối lúa (V_bulk)     : {bulk_volume_mm3:,.2f} mm³")
    print(f"  🔥 TỶ SỐ THỂ TÍCH THUẦN (V_b/V_g) : {pure_volume_ratio:.2f} (Đưa trực tiếp vào train/dự đoán Hồi quy)")
    print("=" * 75)

    if len(lengths) >= 3:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes[0, 0].hist(lengths, bins=15, color='#2196F3', edgecolor='black', alpha=0.7)
        axes[0, 0].axvline(np.mean(lengths), color='red', linestyle='--', linewidth=2)
        axes[0, 0].set_title("Chiều Dài (2a - mm)")

        axes[0, 1].hist(widths, bins=15, color='#4CAF50', edgecolor='black', alpha=0.7)
        axes[0, 1].axvline(np.mean(widths), color='red', linestyle='--', linewidth=2)
        axes[0, 1].set_title("Chiều Rộng (2b - mm)")

        axes[1, 0].hist(areas, bins=15, color='#FF9800', edgecolor='black', alpha=0.7)
        axes[1, 0].axvline(np.mean(areas), color='red', linestyle='--', linewidth=2)
        axes[1, 0].set_title("Diện Tích (mm2)")

        axes[1, 1].hist(vols, bins=15, color='#9C27B0', edgecolor='black', alpha=0.7)
        axes[1, 1].axvline(mean_whole_grain_vol, color='red', linestyle='--', linewidth=2, label=f'Mean={mean_whole_grain_vol:.2f}')
        axes[1, 1].set_title("Thể Tích (mm3)")
        axes[1, 1].legend()
        plt.tight_layout()
        plt.show()


In [ ]:
# Bước 11: GALLERY TRỰC QUAN — TỔNG HỢP ẢNH HẠT QUA TẤT CẢ CÁC GIAI ĐOẠN LƯU
import glob

def show_grain_gallery(folder, title, max_n=8):
    imgs = sorted(glob.glob(os.path.join(folder, "*.png")))[:max_n]
    if not imgs:
        print(f"(Thư mục trống: {folder})")
        return
    n = len(imgs)
    cols = min(4, n)
    rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * 3, rows * 2.8))
    for i, p in enumerate(imgs):
        im = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if im is not None:
            plt.subplot(rows, cols, i + 1)
            if len(im.shape) == 3 and im.shape[2] == 4:
                alpha = im[:, :, 3] / 255.0
                rgb = cv2.cvtColor(im[:, :, :3], cv2.COLOR_BGR2RGB)
                white = np.ones_like(rgb) * 255
                blended = (rgb * alpha[:, :, None] + white * (1 - alpha[:, :, None])).astype(np.uint8)
                plt.imshow(blended)
            else:
                plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            plt.title(os.path.basename(p)[:24], fontsize=8)
            plt.axis('off')
    plt.suptitle(title, fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("🖼️ KIỂM TRA TRỰC QUAN CÁC HẠT ĐÃ ĐƯỢC LƯU:")
show_grain_gallery(DIR_STAGE0_RAW_BEFORE_CLEAN, "Giai đoạn 0: Ảnh Hạt Thô Từ YOLO (TRƯỚC CLEAN)")
show_grain_gallery(DIR_STAGE1_AFTER_CLEAN,      "Giai đoạn 1: Ảnh Hạt Sau Clean 2 Bước (SAU CLEAN)")
show_grain_gallery(DIR_STAGE2_CNN_WHOLE,        "Giai đoạn 2: Hạt NGUYÊN Đạt Chuẩn (Sau CNN)")
show_grain_gallery(DIR_STAGE3_IQR_KEPT,         "Giai đoạn 3: Hạt ĐẠT CHUẨN CUỐI CÙNG (Sau IQR)")


In [ ]:
# ==============================================================================
# BƯỚC 12: NẠP BUNDLE MÔ HÌNH HỒI QUY AI (MACHINE LEARNING)
# ==============================================================================
import joblib

REGRESSION_MODEL_DIR = os.path.join(BASE_PATH, "LINEAR_REGRESSION_MODEL/models/extra_trees")
if not os.path.exists(REGRESSION_MODEL_DIR):
    reg_parent = os.path.join(BASE_PATH, "LINEAR_REGRESSION_MODEL/models")
    if os.path.exists(reg_parent):
        subdirs = [os.path.join(reg_parent, d) for d in os.listdir(reg_parent) if os.path.isdir(os.path.join(reg_parent, d))]
        if subdirs: REGRESSION_MODEL_DIR = subdirs[0]

print(f"📦 Đang nạp mô hình Hồi quy từ: {REGRESSION_MODEL_DIR}")

def load_regression_bundle(model_dir):
    model_path = os.path.join(model_dir, "model.joblib")
    scaler_path = os.path.join(model_dir, "scaler.joblib")
    schema_path = os.path.join(model_dir, "feature_schema.json")
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Không tìm thấy model.joblib tại: {model_dir}")
    
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path) if os.path.exists(scaler_path) else None
    EXPECTED_31_FEATURES = [
        'Bulk_Rice_Volume_mm3', 'Rice_Height_mm', 'Weight_g', 'Empty_Height_mm',
        'Pixels_Per_mm', 'Container_Detected_Diam_px', 'Inner_Diameter_mm', 'Container_Height_mm',
        'Whole_Grains_Count', 'Uniformity_Rate_Pct', 'Estimated_Total_Seeds_Hybrid',
        'Grain_Length_mm_Mean', 'Grain_Length_mm_Min', 'Grain_Length_mm_Max', 'Grain_Length_mm_Std',
        'Grain_Width_mm_Mean', 'Grain_Width_mm_Min', 'Grain_Width_mm_Max', 'Grain_Width_mm_Std',
        'Grain_Thickness_mm_Mean', 'Grain_Thickness_mm_Min', 'Grain_Thickness_mm_Max', 'Grain_Thickness_mm_Std',
        'Grain_Area_mm2_Mean', 'Grain_Area_mm2_Min', 'Grain_Area_mm2_Max', 'Grain_Area_mm2_Std',
        'Grain_Volume_mm3_Mean', 'Grain_Volume_mm3_Min', 'Grain_Volume_mm3_Max', 'Grain_Volume_mm3_Std'
    ]
    ordered_features = EXPECTED_31_FEATURES
    if os.path.exists(schema_path):
        with open(schema_path, 'r', encoding='utf-8') as f:
            schema = json.load(f)
            loaded_feats = schema.get('features', schema.get('ordered_features', []))
            if loaded_feats and len(loaded_feats) == 31:
                ordered_features = loaded_feats
            
    return {'model': model, 'scaler': scaler, 'ordered_features': ordered_features}

regression_bundle = load_regression_bundle(REGRESSION_MODEL_DIR)
print(f"✅ Nạp thành công mô hình Hồi quy: {regression_bundle['model'].__class__.__name__} ({len(regression_bundle['ordered_features'])} features)")


In [ ]:
# ==============================================================================
# BƯỚC 13: TRÍCH XUẤT 31 ĐẶC TRƯNG HỒI QUY (VỚI TỶ SỐ THỂ TÍCH THUẦN)
# ==============================================================================
def assemble_features_v2(bundle, records, metrics, bulk_vol, rice_h, weight_g, empty_h, px_per_mm, inner_diam, cont_height):
    lens_arr = np.array([float(m["length_mm"]) for m in metrics])
    wids_arr = np.array([float(m["width_mm"]) for m in metrics])
    thks_arr = np.array([float(m["thickness_mm"]) for m in metrics])
    area_arr = np.array([float(m["area_mm2"]) for m in metrics])
    vols_arr = np.array([float(m["volume_mm3"]) for m in metrics])

    def calc_stats(arr):
        return {
            "Mean": round(float(np.mean(arr)), 3),
            "Min": round(float(np.min(arr)), 3),
            "Max": round(float(np.max(arr)), 3),
            "Std": round(float(np.std(arr, ddof=0)), 3)
        }

    lens_st = calc_stats(lens_arr)
    wids_st = calc_stats(wids_arr)
    thks_st = calc_stats(thks_arr)
    area_st = calc_stats(area_arr)
    vols_st = calc_stats(vols_arr)

    mean_vol = vols_st["Mean"]
    batch_unif = evaluate_batch_uniformity(vols_arr.tolist())
    unif_rate = round(float(batch_unif.get("uniformity_rate_pct", 0.0)), 2)

    # 🔥 TỶ SỐ THỂ TÍCH THUẦN: V_bulk / mean_vol (Không nhân bất kỳ hệ số nào)
    volume_ratio_feat = round(bulk_vol / mean_vol, 2) if mean_vol > 0 else 0.0

    feat_dict = {
        "Bulk_Rice_Volume_mm3": round(bulk_vol, 2),
        "Rice_Height_mm": round(float(rice_h), 2),
        "Weight_g": float(weight_g),
        "Empty_Height_mm": float(empty_h),
        "Pixels_Per_mm": round(float(px_per_mm), 2),
        "Container_Detected_Diam_px": round(float(container_info.get("inner_w_px", 0)), 2),
        "Inner_Diameter_mm": float(inner_diam),
        "Container_Height_mm": float(cont_height),
        "Whole_Grains_Count": float(len(records)),
        "Uniformity_Rate_Pct": unif_rate,
        "Estimated_Total_Seeds_Hybrid": float(volume_ratio_feat),
        "Grain_Length_mm_Mean": lens_st["Mean"],
        "Grain_Length_mm_Min": lens_st["Min"],
        "Grain_Length_mm_Max": lens_st["Max"],
        "Grain_Length_mm_Std": lens_st["Std"],
        "Grain_Width_mm_Mean": wids_st["Mean"],
        "Grain_Width_mm_Min": wids_st["Min"],
        "Grain_Width_mm_Max": wids_st["Max"],
        "Grain_Width_mm_Std": wids_st["Std"],
        "Grain_Thickness_mm_Mean": thks_st["Mean"],
        "Grain_Thickness_mm_Min": thks_st["Min"],
        "Grain_Thickness_mm_Max": thks_st["Max"],
        "Grain_Thickness_mm_Std": thks_st["Std"],
        "Grain_Area_mm2_Mean": area_st["Mean"],
        "Grain_Area_mm2_Min": area_st["Min"],
        "Grain_Area_mm2_Max": area_st["Max"],
        "Grain_Area_mm2_Std": area_st["Std"],
        "Grain_Volume_mm3_Mean": vols_st["Mean"],
        "Grain_Volume_mm3_Min": vols_st["Min"],
        "Grain_Volume_mm3_Max": vols_st["Max"],
        "Grain_Volume_mm3_Std": vols_st["Std"]
    }
    return pd.DataFrame([feat_dict], columns=bundle["ordered_features"])

features_df = assemble_features_v2(
    bundle=regression_bundle,
    records=whole_records,
    metrics=whole_grain_metrics_list,
    bulk_vol=bulk_volume_mm3,
    rice_h=container_info["rice_height_mm"],
    weight_g=INPUT_WEIGHT_G,
    empty_h=EMPTY_HEIGHT_MM,
    px_per_mm=pixels_per_mm,
    inner_diam=INNER_DIAM_MM,
    cont_height=CONTAINER_HEIGHT_MM
)
print("✅ Trích xuất 31 features thành công!")
print(f"• Tỷ số thể tích thuần (V_b / V_g): {features_df['Estimated_Total_Seeds_Hybrid'].iloc[0]:.2f}")


In [ ]:
# ==============================================================================
# BƯỚC 14: DỰ ĐOÁN SỐ LƯỢNG HẠT BẰNG HỒI QUY AI (+ THAM CHIẾU CÂN NẶNG)
# ==============================================================================
model = regression_bundle["model"]
scaler = regression_bundle["scaler"]

if scaler is not None:
    scaler_input = features_df if hasattr(scaler, "feature_names_in_") else features_df.to_numpy()
    X_scaled = scaler.transform(scaler_input)
else:
    X_scaled = features_df.to_numpy()

raw_pred = float(model.predict(X_scaled).item())
ai_predicted_seeds = max(0, int(round(raw_pred)))

# Tham chiếu Cân nặng (TGW ~ 26g/1000 hạt -> 1 hạt ~ 0.026g)
seed_weight_ref = 0.026
weight_based_seeds = int(round(INPUT_WEIGHT_G / seed_weight_ref)) if INPUT_WEIGHT_G > 0 else None

print("=" * 75)
print("  🤖 KẾT QUẢ DỰ ĐOÁN RICE VISION AI PIPELINE V2")
print("=" * 75)
print(f"  🎯 SỐ HẠT DỰ ĐOÁN HỒI QUY (Chính)   : {ai_predicted_seeds} HẠT (Raw: {raw_pred:.2f})")
if weight_based_seeds is not None:
    print(f"  ⚖️  ƯỚC LƯỢNG THEO CÂN NẶNG (Tham chiếu): {weight_based_seeds} HẠT ({INPUT_WEIGHT_G}g / {seed_weight_ref}g)")
print(f"  📐 Tỷ số Thể tích thuần (V_b/V_g)    : {features_df['Estimated_Total_Seeds_Hybrid'].iloc[0]:.2f}")
print(f"  🌾 Số hạt nguyên bề mặt đo được      : {len(whole_records)} hạt")
print("=" * 75)


In [ ]:
# ==============================================================================
# BƯỚC 15: BÁO CÁO TỔNG HỢP VÀ XUẤT TẬP TIN
# ==============================================================================
summary_data = {
    "Image_Name": os.path.basename(IMAGE_PATH),
    "Bulk_Rice_Volume_mm3": round(bulk_volume_mm3, 2),
    "Mean_Grain_Volume_mm3": round(mean_whole_grain_vol, 2) if mean_whole_grain_vol else None,
    "Pure_Volume_Ratio": round(pure_volume_ratio, 2) if pure_volume_ratio else None,
    "Weight_g": INPUT_WEIGHT_G,
    "Whole_Grains_Count": len(whole_records),
    "Predicted_Seeds_AI": ai_predicted_seeds,
    "Weight_Based_Seeds": weight_based_seeds,
    "YOLO_Model": os.path.basename(YOLO_MODEL_PATH),
    "Regression_Model": regression_bundle["model"].__class__.__name__
}

df_summary = pd.DataFrame([summary_data])
summary_csv = os.path.join(OUTPUT_REPORT_DIR, "pipeline_v2_prediction_summary.csv")
df_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

features_csv = os.path.join(OUTPUT_REPORT_DIR, "pipeline_v2_features.csv")
features_df.to_csv(features_csv, index=False, encoding="utf-8-sig")

print(f"💾 Đã lưu báo cáo tại : {summary_csv}")
print(f"💾 Đã lưu 31 features tại: {features_csv}")
print(f"📁 Thư mục ảnh từng hạt : {OUTPUT_GRAIN_CROPS_ROOT}")
display(df_summary)
